In [7]:
"""
Evaluation for external_pred_corrected.csv (from correct_external_predictions.py)
-- the out-of-domain Metamorphosis test set, corrected version.

Runs the same four complementary views as evaluate_oof_corrected.ipynb, once
for ALL LANGUAGES POOLED and once per individual language, all in one pass:
  1. Token-level  - is the label right per token? (ignores span boundaries)
  2. Span-level   - is the full span (start, end, label) right? (seqeval strict)
  3. Span-boundary-only - did the model find the span at all, ignoring label?
  4. BIO-prefix accuracy - B / I / O accuracy regardless of the event label
plus a confusion matrix for each.

Each report is computed on its own subset (pooled, or one language) treated
as one corpus -- no per-sentence averaging.

NOTE on file structure: word-level, one row per word. Column names differ
slightly from the k-fold OOF file -- 'gold_label' here (not 'true_label'),
plus a 'language' column since this file spans multiple languages. Sentences
are reconstructed by grouping on (language, source_file, sentence_id),
ordered by word_index.
"""

from itertools import chain

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

from seqeval.metrics import classification_report as seq_report
from seqeval.scheme import IOB2
from sklearn.metrics import (
    classification_report as sk_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
)

# -- Load ----------------------------------------------------------------
path = "/Users/noavisser/Documents/event_annotation/classification/habrok/event_crf_predictions/external_predictions"
MODEL_FILE_TAG = "mmBERT-base"  # e.g. "mmBERT-base" or "xlm-roberta-base" -- must match correct_external_predictions.py's --base_model_short_name
df_all = pd.read_csv(f"{path}/external_pred_corrected_{MODEL_FILE_TAG}.csv")
# columns: language, source_file, sentence_id, word_index, word,
#          gold_label, pred_label_raw, pred_label_corrected,
#          correction_applied, correct

PRED_COL = "pred_label_corrected"  # swap for 'pred_label_raw' to check uncorrected predictions
EVENT_LABELS = ["process", "stative_event", "change_of_state", "non_event"]

languages = sorted(df_all["language"].unique())
print(f"Loaded {len(df_all)} word-level rows, languages found: {languages}")
print(f"Correction applied: {df_all['correction_applied'].unique().tolist()}")


Loaded 4293 word-level rows, languages found: ['BI', 'Dutch', 'Italian']
Correction applied: ['all_B']


In [8]:
# -- Helper functions, shared across pooled + every per-language run -------

def strip_bio(label):
    return 'O' if label == 'O' else label.split('-', 1)[1]

def strip_suffix(label):
    return "O" if label == "O" else label.split("-")[0]

def flat_to_bio_binary(labels):
    return ["O" if l == "O" else l.split("-")[0] + "-EVENT" for l in labels]

def safe_seq_report(gold_seqs, pred_seqs, **kwargs):
    """seqeval's classification_report crashes (ValueError: max() arg is empty)
    if a subset has literally zero labeled entities of any kind -- common for
    a single language on a short test document. Guard rather than let one
    thin language kill the whole run."""
    try:
        return seq_report(gold_seqs, pred_seqs, **kwargs)
    except ValueError:
        return "(no entities found in gold or predictions for this subset -- nothing to report)"

def evaluate_subset(df, label):
    """Runs all four views + confusion matrix for one subset (pooled, or one language)."""
    print("\n" + "#" * 70)
    print(f"# {label}")
    print("#" * 70)

    gold_seqs, pred_seqs = [], []
    for _, g in df.sort_values("word_index").groupby(["language", "source_file", "sentence_id"], sort=False):
        gold_seqs.append(g["gold_label"].tolist())
        pred_seqs.append(g[PRED_COL].tolist())

    bad = [i for i, (g, p) in enumerate(zip(gold_seqs, pred_seqs)) if len(g) != len(p)]
    if bad:
        print(f"WARNING: {len(bad)} sentences have mismatched gold/pred lengths -- check these.")
    print(f"{len(gold_seqs)} sentences, {len(df)} words")

    flat_gold = [strip_bio(l) for l in chain.from_iterable(gold_seqs)]
    flat_pred = [strip_bio(l) for l in chain.from_iterable(pred_seqs)]

    print("\n-- 1. TOKEN-LEVEL --")
    print(sk_report(flat_gold, flat_pred, labels=EVENT_LABELS + ["O"], zero_division=0, digits=3))

    print("-- 2. SPAN-LEVEL, seqeval default (conlleval-style, no scheme enforcement) --")
    print(safe_seq_report(gold_seqs, pred_seqs, zero_division=0, digits=3))

    gold_bin = [flat_to_bio_binary(seq) for seq in gold_seqs]
    pred_bin = [flat_to_bio_binary(seq) for seq in pred_seqs]
    print("-- 3. SPAN-BOUNDARY ONLY, seqeval default (all event types merged -> EVENT) --")
    print(safe_seq_report(gold_bin, pred_bin, zero_division=0, digits=3))

    gold_bio_flat = [strip_suffix(l) for l in chain.from_iterable(gold_seqs)]
    pred_bio_flat = [strip_suffix(l) for l in chain.from_iterable(pred_seqs)]
    print("-- 4. B / I / O PREFIX ACCURACY --")
    print(sk_report(gold_bio_flat, pred_bio_flat, labels=["B", "I", "O"], zero_division=0, digits=3))

    cm_labels = EVENT_LABELS + ["O"]
    cm = confusion_matrix(flat_gold, flat_pred, labels=cm_labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cm_labels)
    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
    plt.title(f"Token-level confusion matrix -- external (Metamorphosis)\n{label}")
    plt.tight_layout()
    tag = label.replace(" ", "_").replace("=", "")
    plt.savefig(f"confusion_matrix_external_{tag}.png", dpi=150)
    plt.show()


In [9]:
evaluate_subset(df_all, "ALL LANGUAGES POOLED")



######################################################################
# ALL LANGUAGES POOLED
######################################################################
196 sentences, 4293 words

-- 1. TOKEN-LEVEL --
                 precision    recall  f1-score   support

        process      0.783     0.801     0.792      1506
  stative_event      0.690     0.712     0.701       715
change_of_state      0.484     0.437     0.459       103
      non_event      0.866     0.843     0.854      1399
              O      0.916     0.898     0.907       570

       accuracy                          0.804      4293
      macro avg      0.748     0.738     0.743      4293
   weighted avg      0.805     0.804     0.804      4293

-- 2. SPAN-LEVEL, seqeval default (conlleval-style, no scheme enforcement) --
                 precision    recall  f1-score   support

change_of_state      0.333     0.364     0.348        11
      non_event      0.754     0.731     0.742       201
        process     

/var/folders/3x/ckfzdwcj68lfxmnrhrbm4hsw0000gn/T/ipykernel_11603/4225191384.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
for lang in languages:
    evaluate_subset(df_all[df_all["language"] == lang], f"LANGUAGE = {lang}")



######################################################################
# LANGUAGE = BI
######################################################################
67 sentences, 1410 words

-- 1. TOKEN-LEVEL --
                 precision    recall  f1-score   support

        process      0.789     0.846     0.817       397
  stative_event      0.775     0.755     0.765       306
change_of_state      0.871     0.659     0.750        41
      non_event      0.908     0.900     0.904       449
              O      0.905     0.876     0.890       217

       accuracy                          0.843      1410
      macro avg      0.849     0.807     0.825      1410
   weighted avg      0.844     0.843     0.842      1410

-- 2. SPAN-LEVEL, seqeval default (conlleval-style, no scheme enforcement) --
                 precision    recall  f1-score   support

change_of_state      0.400     0.400     0.400         5
      non_event      0.791     0.726     0.757        73
        process      0.500  

/var/folders/3x/ckfzdwcj68lfxmnrhrbm4hsw0000gn/T/ipykernel_11603/4225191384.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/3x/ckfzdwcj68lfxmnrhrbm4hsw0000gn/T/ipykernel_11603/4225191384.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



######################################################################
# LANGUAGE = Italian
######################################################################
64 sentences, 1376 words

-- 1. TOKEN-LEVEL --
                 precision    recall  f1-score   support

        process      0.768     0.730     0.749       526
  stative_event      0.483     0.840     0.613       119
change_of_state      0.486     0.353     0.409        51
      non_event      0.876     0.780     0.825       508
              O      0.894     0.936     0.915       172

       accuracy                          0.770      1376
      macro avg      0.702     0.728     0.702      1376
   weighted avg      0.789     0.770     0.773      1376

-- 2. SPAN-LEVEL, seqeval default (conlleval-style, no scheme enforcement) --
                 precision    recall  f1-score   support

change_of_state      0.500     0.400     0.444         5
      non_event      0.693     0.675     0.684        77
        process      0.

/var/folders/3x/ckfzdwcj68lfxmnrhrbm4hsw0000gn/T/ipykernel_11603/4225191384.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# -- Build a LaTeX results table (span-level F1 + support per class per language) --

CLASS_ORDER = [
    ("non_event", "Non-Event"),
    ("stative_event", "Stative"),
    ("process", "Process"),
    ("change_of_state", "Change of state"),
]
LANGUAGE_ORDER = ["Dutch", "BI", "Italian"]  # edit order/set here if you want a different arrangement
LANGUAGE_DISPLAY = {"Dutch": "Dutch", "BI": "Indonesian", "Italian": "Italian"}

def get_span_level_dict(df):
    """Same span-level (seqeval default) computation as evaluate_subset's view 2,
    but returning a dict (output_dict=True) instead of a printed string, so the
    numbers can be pulled out programmatically for the LaTeX table below."""
    gold_seqs, pred_seqs = [], []
    for _, g in df.sort_values("word_index").groupby(["language", "source_file", "sentence_id"], sort=False):
        gold_seqs.append(g["gold_label"].tolist())
        pred_seqs.append(g[PRED_COL].tolist())
    try:
        return seq_report(gold_seqs, pred_seqs, zero_division=0, output_dict=True)
    except ValueError:
        return None  # no entities found in this subset

def build_latex_table(df_all, model_name, repeat_header_footer=True):
    """repeat_header_footer=True reprints the header row just before \\bottomrule,
    matching the requested table format -- set False to drop that repeat."""
    header_cols = " & ".join(disp for _, disp in CLASS_ORDER)
    header_row = f"{model_name}    & {header_cols}   & F1 \\\\"

    lines = [
        r"\begin{table}[ht!]\centering",
        r"\begin{tabular}{l rrrr r}\toprule",
        header_row,
        r"\midrule",
    ]
    for lang in LANGUAGE_ORDER:
        d = get_span_level_dict(df_all[df_all["language"] == lang])
        disp_name = LANGUAGE_DISPLAY.get(lang, lang)
        if d is None:
            lines.append(f"{disp_name}       & \\multicolumn{{5}}{{c}}{{no entities found}} \\\\")
            lines.append(r"\midrule")
            continue
        f1_vals = [d.get(key, {}).get("f1-score", 0.0) for key, _ in CLASS_ORDER]
        support_vals = [d.get(key, {}).get("support", 0) for key, _ in CLASS_ORDER]
        overall_f1 = d.get("micro avg", {}).get("f1-score", 0.0)
        overall_support = d.get("micro avg", {}).get("support", 0)

        f1_str = " & ".join(f"{v:.3f}" for v in f1_vals)
        lines.append(f"{disp_name}       & {f1_str}          & {overall_f1:.3f}    \\\\")
        support_str = " & ".join(str(v) for v in support_vals)
        lines.append(f"Support     & {support_str}       &  {overall_support} \\\\")
        lines.append(r"\midrule")

    if repeat_header_footer:
        lines.append(header_row)
        lines.append(r"\midrule")

    lines.append(r"\bottomrule\end{tabular}")
    lines.append(
        r"\caption{Event classification results (span level) on an excerpt of Metamorphosis using the "
        + model_name + r" crosslingual for each language.}\label{metamorphosisresults}"
    )
    lines.append(r"\end{table}")
    return "\n".join(lines)


In [13]:
MODEL_NAME = "mmBERT"  # display label for the table -- set independently of MODEL_FILE_TAG if you want a different caption name

table_latex = build_latex_table(df_all, MODEL_NAME)
print(table_latex)


\begin{table}[ht!]\centering
\begin{tabular}{l rrrr r}\toprule
mmBERT    & Non-Event & Stative & Process & Change of state   & F1 \\
\midrule
Dutch       & 0.808 & 0.596 & 0.688 & 0.000          & 0.703    \\
Support     & 51 & 29 & 61 & 1       &  142 \\
\midrule
Indonesian       & 0.757 & 0.489 & 0.489 & 0.400          & 0.585    \\
Support     & 73 & 51 & 71 & 5       &  200 \\
\midrule
Italian       & 0.684 & 0.390 & 0.648 & 0.444          & 0.628    \\
Support     & 77 & 12 & 71 & 5       &  165 \\
\midrule
mmBERT    & Non-Event & Stative & Process & Change of state   & F1 \\
\midrule
\bottomrule\end{tabular}
\caption{Event classification results (span level) on an excerpt of Metamorphosis using the mmBERT crosslingual for each language.}\label{metamorphosisresults}
\end{table}
